# ✈️ Flight Delay Analysis — Tech Challenge Fase 3

**Notebook de exploração interativa** — complementa os módulos Python do pipeline.

Este notebook cobre:
1. Setup e carregamento de dados
2. Análise Exploratória (EDA)
3. Modelagem supervisionada
4. Clusterização e PCA
5. Detecção de anomalias
6. Conclusões

---
> **Pré-requisito**: coloque o arquivo `flights.csv` na pasta `data/`

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='viridis')

print('Bibliotecas carregadas com sucesso!')

## 1. Carregamento e Pré-processamento

In [ ]:
from data_preprocessing import preprocess_pipeline

# Usar SAMPLE_FRAC=0.05 para prototipagem rápida
# Aumentar para 1.0 para treinamento final
df = preprocess_pipeline(sample_frac=1.0)

print(f'Shape: {df.shape}')
print(f'Período: {df["MONTH"].min()} a {df["MONTH"].max()} (mês)')
print(f'Taxa de atraso (>=15 min): {df["DELAYED"].mean():.2%}')
df.head()

In [ ]:
# Estatísticas descritivas
cols_stats = ['ARRIVAL_DELAY', 'DEPARTURE_DELAY', 'DISTANCE', 'SCHEDULED_TIME', 'TAXI_OUT']
df[cols_stats].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2)

## 2. EDA — Visualizações

In [ ]:
from eda import run_full_eda
run_full_eda(df)

In [ ]:
# Distribuição do ARRIVAL_DELAY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data = df['ARRIVAL_DELAY'].clip(-60, 300)
axes[0].hist(data, bins=80, color='#4C72B0', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', label='0 min')
axes[0].axvline(15, color='orange', linestyle='--', label='15 min (threshold)')
axes[0].set_title('Distribuição do Atraso na Chegada')
axes[0].set_xlabel('Atraso (min)')
axes[0].legend()

counts = df['DELAYED'].value_counts()
axes[1].pie(counts, labels=['Não Atrasado', 'Atrasado'], autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Proporção de Voos Atrasados')
plt.tight_layout()
plt.show()

In [ ]:
# Atraso por dia da semana e período do dia
day_names = {1:'Seg',2:'Ter',3:'Qua',4:'Qui',5:'Sex',6:'Sáb',7:'Dom'}
df2 = df.copy()
df2['DAY_NAME'] = df2['DAY_OF_WEEK'].map(day_names)

pivot = (df2.groupby(['DAY_NAME','PERIOD_OF_DAY'])['ARRIVAL_DELAY']
           .mean().unstack('PERIOD_OF_DAY'))

day_order = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
period_order = ['MADRUGADA','MANHA','TARDE','NOITE']
pivot = pivot.reindex(index=day_order,
                      columns=[c for c in period_order if c in pivot.columns])

plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label':'Atraso médio (min)'})
plt.title('Atraso Médio: Dia da Semana × Período do Dia')
plt.show()

## 3. Modelagem Supervisionada

In [ ]:
from supervised_models import run_supervised
clf_results, reg_results = run_supervised(df)

print('\n--- Classificação ---')
display(clf_results)
print('\n--- Regressão ---')
display(reg_results)

## 4. Clusterização e PCA

In [ ]:
from unsupervised_models import run_unsupervised
km, labels = run_unsupervised(df)

## 5. Detecção de Anomalias

In [ ]:
from anomaly_detection import run_anomaly_detection
df_anomaly = run_anomaly_detection(df)

print(f'Voos anômalos: {df_anomaly["IS_ANOMALY"].sum():,}')
print(f'Atraso médio — Normal: {df_anomaly[df_anomaly["IS_ANOMALY"]==0]["ARRIVAL_DELAY"].mean():.1f} min')
print(f'Atraso médio — Anomalia: {df_anomaly[df_anomaly["IS_ANOMALY"]==1]["ARRIVAL_DELAY"].mean():.1f} min')

## 6. Conclusões

### Principais Achados

1. **`DEPARTURE_DELAY`** é o preditor mais forte de `ARRIVAL_DELAY` — atrasos na partida propagam para chegada.
2. **Sexta-feira à noite** apresenta os maiores atrasos médios.
3. **Junho/Julho e Dezembro** são os meses com maior taxa de atraso (férias + clima).
4. **`LATE_AIRCRAFT_DELAY`** é a causa mais frequente — efeito cascata operacional.
5. **K-Means** identificou perfis distintos de rotas: domésticas curtas vs. transcontinentais.
6. **Isolation Forest** detectou ~5% de voos com comportamento atípico.

### Limitações
- Dataset de 2015 apenas (sem generalização temporal)
- Ausência de dados meteorológicos detalhados
- `DEPARTURE_DELAY` pode ser data leakage em cenário de predição pré-partida

### Próximos Passos
- Integração com API meteorológica (NOAA)
- Modelo de séries temporais (LSTM) para efeito cascata
- Dashboard interativo com Streamlit/Dash
- Deploy com FastAPI + Docker